# Optimization Campaign (HITL)

**Prerequisites:** TermNorm backend at `http://127.0.0.1:8000` | Groq API key in `.env` | Restart kernel after first sync

**Workflow:** Setup → Data → Explore → Optimize → Results

## 1. Setup

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import json
from _campaign_lib import *

svc = await init_services()
TASK_DESCRIPTION = load_task_description(
    r"C:\Users\dsacc\OfficeAddinApps\TermNorm-excel\backend-api\config\LCA_INPUT_PATTERNS.md"
)

2026-03-13 15:40:54 INFO     [httpx] HTTP Request: GET http://127.0.0.1:8000/pipeline "HTTP/1.1 200 OK"
2026-03-13 15:40:54 INFO     [api.services.pipeline_discovery] Matched known pipeline 'termnorm'; using enriched schema
2026-03-13 15:40:54 INFO     [api.services.campaign.campaign_init] Pipeline schema loaded: termnorm vv1.1


Experiment : production_historical
Mappings   : 887 total, 812 with verified ground truth
Queries    : 40  |  Session terms: 93
Loaded task description: 3751 chars from LCA_INPUT_PATTERNS.md


In [3]:
campaign_config = {
    "sample_size": 15,              # queries per eval step (service default: all)
    "exploration_rate": 0.5,             # PRIMARY KNOB: 0.0=conservative, 1.0=aggressive
    "improvement_areas": "profile schema quality, web search relevance",
    "exclude_steps": ["llm_ranking"],    # steps to skip (e.g. ["entity_profiling"])
    "pipeline_overrides": {},
    "optimization": {
        "patience": 2,                   # default: 3
        "max_rounds": 3,                 # default: 10
    },
    "eval_llm": {
        # --- Groq (free tier, open-source models) ---
        "model": "openai/gpt-oss-120b",
        # "model": "moonshotai/kimi-k2-instruct-0905"
        "provider_url": "https://api.groq.com/openai/v1/chat/completions",
        # --- Anthropic (cost: opus >> sonnet >> haiku) ---
        # "model": "claude-opus-4-6",          # best quality
        # "model": "claude-sonnet-4-6",      # good balance
        # "model": "claude-haiku-4-5-20251001",  # cheapest
        # "provider_url": "https://api.anthropic.com",
        "max_tokens": 2000,              # response length budget
    },
    "grid_search": {
        "context": "A terminology normalization pipeline that matches raw material "
                    "descriptions to standardized database terms using entity profiling "
                    "and candidate ranking.",
        "grid_budget": 35,               # default: 0 (full grid)
        "sample_size": 6,     # default: 0 (all queries)
        "shared_queries": False,          # default: True
    },
}

In [4]:
#@title Pipeline snapshot (full config for reproducibility)
pipeline_config_full = await show_pipeline_snapshot(svc)

2026-03-13 15:40:54 INFO     [httpx] HTTP Request: GET http://127.0.0.1:8000/pipeline "HTTP/1.1 200 OK"


  PIPELINE SNAPSHOT: TermNorm v1.1
  Nodes:   ['fuzzy_matching', 'web_search', 'entity_profiling', 'token_matching', 'llm_ranking', 'direct_prompt']
  Schemas: ['entity_profile/1', 'llm_ranking_output/1']
  Prompts: ['entity_profiling/1', 'llm_ranking/1']

{
  "name": "TermNorm",
  "version": "v1.1",
  "available_models": [
    "moonshotai/kimi-k2-instruct-0905",
    "meta-llama/llama-4-scout-17b-16e-instruct",
    "moonshotai/kimi-k2-instruct",
    "openai/gpt-oss-120b"
  ],
  "nodes": {
    "fuzzy_matching": {
      "type": "DeterministicFunction",
      "config": {
        "threshold": 70,
        "scorer": "WRatio",
        "limit": 5
      }
    },
    "web_search": {
      "type": "ExternalService",
      "config": {
        "max_sites": 7,
        "num_results": 20,
        "content_char_limit": 800,
        "url_fetch_multiplier": 2,
        "fallback_keywords_limit": 8,
        "query_prefix": "",
        "query_suffix": "",
        "brave_api_timeout": 10,
        "scrape_tim

In [5]:
#@title Build pipeline params
pipeline_params = configure_pipeline(svc, campaign_config)

Active steps: ['cache_lookup', 'fuzzy_matching', 'web_search', 'entity_profiling', 'token_matching']
  Excluded: ['llm_ranking']


## 2. Data

In [6]:
#@title Load datasets
# Set EXCEL_PATH to load from BOM-example.xlsx; leave empty to use stored data
EXCEL_PATH = r"C:\Users\dsacc\Desktop\project-TermNorm\OneDrive_2025-07-02\Austausch Beispiele\Prozessnamen\BOM-example.xlsx"  # e.g. "../data/BOM-example.xlsx"
FORCE_RELOAD = False  # Set True to re-read Excel and overwrite stored datasets

train_data, svc["session_terms"] = prepare_datasets(
    svc["store"], svc["backend_id"],
    excel_path=EXCEL_PATH or None,
    force=FORCE_RELOAD,
)


  Train              : 984 queries
  Test (processes)   : 82 queries
  Test (material)    : 165 queries
  ------------------------------------------------
  Combined queries   : 820 (deduplicated)
  Session identifiers: 94 unique targets


In [7]:
#@title Prepare evaluation context
campaign_rounds = []
baseline_results = []

baseline, eval_data, backend_status = await prepare_eval_context(
    svc, train_data,
)

RUN_BASELINE = False  # Set True to evaluate baseline before exploration
if RUN_BASELINE:
    campaign_rounds, baseline_results = await run_baseline_eval(
        baseline, eval_data, campaign_config, svc,
    )

2026-03-13 15:40:55 INFO     [httpx] HTTP Request: GET http://127.0.0.1:8000/status "HTTP/1.1 200 OK"



BACKEND STATUS
  Session Active                 True
  Active Sessions                1
  Terms Loaded                   94
  Match Database Identifiers     110
  Match Database Aliases         694
  Experiments Count              4
  Mappings Count                 1126
  Pipeline Version               v1.1
  Llm Provider                   groq
  Llm Model                      moonshotai/kimi-k2-instruct-0905
  ------------------------------------------------
  Experiments                   
    0_production_realtime        0 mappings
    1_production_historical      887 mappings
    2_bom_materials              159 mappings
    3_bom_processing             80 mappings

Evaluation data: 984 queries


In [8]:
#@title Candidate coverage (post-eval diagnostic)
cov_df = run_coverage_diagnostic(
    baseline_results,
    svc["store"], svc["backend_id"], svc["experiment_id"],
)

Loaded 40 eval queries
Eval runs: 72 completed runs, 20 in-progress
  run_id                name           model                      temp  accuracy  queries
  scan_15a5c1e9         scan                                      0.0   50.0%     6      
  scan_86f17bab         scan                                      0.0   66.7%     6      
  scan_fcb7bf9b         scan                                      0.0   50.0%     6      
  scan_01c3382c         scan                                      0.0   66.7%     6      
  scan_7d0c905a         scan                                      0.0   33.3%     6      
  scan_e9f03615         scan                                      0.0   66.7%     6      
  scan_3aff5881         scan                                      0.0   33.3%     6      
  scan_39b9fc27         scan                                      0.0   50.0%     6      
  scan_dff96710         scan                                      0.0   33.3%     6      
  scan_3dbc066d         scan    

In [ ]:
#@title Campaign explorer (overview / detail / diff)
# Overview: see all campaigns
list_campaigns(store=svc["store"], backend_id=svc["backend_id"])

# Detail: uncomment and paste a campaign_id
# list_campaigns(store=svc["store"], backend_id=svc["backend_id"],
#                campaign_id="cycle_68e2c53845c3")

# Diff: compare current config vs stored campaign
# diff_campaign_config(svc["store"], svc["backend_id"],
#                      "cycle_68e2c53845c3", campaign_config,
#                      pipeline_params=pipeline_params)

## 3. Explore

Two exploration paths: **Smart Search** (scan advisor + sensitivity scan) or **Grid Search** (brute-force sweep). Use one or both.

### 3a. Smart Search

In [9]:
#@title Browse variant library
display_variant_library()
# Filter examples:
# display_variant_library(source="PromptWizard")
# display_variant_library(axes=["thinking_style", "persona"])

Variant Library
  Sources: PromptPotter (16), PromptWizard (40)

  persona (6 variants — PromptPotter: 4, PromptWizard: 2)
    [ 0] [PromptPotter] (empty baseline)
    [ 1] [PromptPotter] You are a domain expert with deep knowledge of this field.
    [ 2] [PromptPotter] You are a precise, analytical system that evaluates candidates methodi...
    [ 3] [PromptPotter] You are a careful assistant that considers all options before deciding...
    [ 4] [PromptWizard 2024] You are a specialist who evaluates candidates based on domain-specific...
    [ 5] [PromptWizard 2024] You are a meticulous researcher who cross-references information befor...

  task_intent (4 variants — PromptPotter: 4)
    [ 0] [PromptPotter] (empty baseline)
    [ 1] [PromptPotter] Your task is to identify the single best match from the candidates.
    [ 2] [PromptPotter] Rank candidates by how well they match the concept described.
    [ 3] [PromptPotter] Evaluate each candidate for semantic equivalence to the query 

{'prompt_fields': {'persona': [{'text': '', 'source': 'PromptPotter'},
   {'text': 'You are a domain expert with deep knowledge of this field.',
    'source': 'PromptPotter'},
   {'text': 'You are a precise, analytical system that evaluates candidates methodically.',
    'source': 'PromptPotter'},
   {'text': 'You are a careful assistant that considers all options before deciding.',
    'source': 'PromptPotter'},
   {'text': 'You are a specialist who evaluates candidates based on domain-specific criteria and professional standards.',
    'source': 'PromptWizard',
    'year': 2024},
   {'text': 'You are a meticulous researcher who cross-references information before making judgments.',
    'source': 'PromptWizard',
    'year': 2024}],
  'task_intent': [{'text': '', 'source': 'PromptPotter'},
   {'text': 'Your task is to identify the single best match from the candidates.',
    'source': 'PromptPotter'},
   {'text': 'Rank candidates by how well they match the concept described.',
    'so

In [10]:
# preview_advisor_prompt()
preview_advisor_prompt(campaign_config, svc, task_description="TASK_DESCRIPTION", raw=True)

2026-03-13 15:40:55 INFO     [api.services.search.smart_search] filter_variant_library: dropped all prompt_fields (llm_ranking not active)


You are an expert prompt optimization advisor. Recommend which axes (parameters and prompt fields) to prioritize in a sensitivity scan.

## Constraints (apply strictly)
- Do NOT recommend *_model axes — place them in axes_to_skip.
- Response must fit within 1500 tokens. Be terse.

## Pipeline: TermNorm AI terminology normalization pipeline
Steps execute sequentially — each step's output feeds the next:
[
  {
    "name": "cache_lookup",
    "node_role": "cache",
    "short_circuit": true
  },
  {
    "name": "fuzzy_matching",
    "node_role": "candidate_source",
    "short_circuit": true
  },
  {
    "name": "web_search",
    "node_role": "enricher"
  },
  {
    "name": "entity_profiling",
    "node_role": "enricher"
  },
  {
    "name": "token_matching",
    "node_role": "candidate_source"
  }
]

## Task Context
TASK_DESCRIPTION
## Tunable Parameters (per step)
[
  {
    "name": "fuzzy_matching",
    "param_keys": [
      "fuzzy_scorer",
      "fuzzy_threshold"
    ]
  },
  {
    "name

In [11]:
#@title Scan advisor
advisory, scan_variants, schema_labels = await run_scan_advisor(
    campaign_config, svc,
    task_description=TASK_DESCRIPTION if "TASK_DESCRIPTION" in dir() else "",
)

2026-03-13 15:40:55 INFO     [api.services.search.smart_search] filter_variant_library: dropped all prompt_fields (llm_ranking not active)


SCAN ADVISOR -- pipeline-aware sensitivity setup
  Pipeline: termnorm (v1.1)
  Steps: ['cache_lookup', 'fuzzy_matching', 'web_search', 'entity_profiling', 'token_matching', 'llm_ranking']
  Excluded: ['llm_ranking']
  Task context: # Domain Context: Life Cycle Assessment (LCA) Terminology

This document capture...
  Calling openai/gpt-oss-120b ...



2026-03-13 15:41:00 INFO     [httpx] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-13 15:41:00 WARNING  [api.services.search.scan_advisor] Scan advisor validation: pipeline_param axis 'entity_profiling_schema' not found in PipelineSchema param_keys: ['content_char_limit', 'fuzzy_scorer', 'fuzzy_threshold', 'max_sites', 'max_token_candidates', 'num_results', 'profiling_max_tokens', 'profiling_model', 'profiling_prompt', 'profiling_schema', 'profiling_temperature', 'query_prefix', 'query_suffix', 'ranking_max_tokens', 'ranking_model', 'ranking_prompt', 'ranking_sample_size', 'ranking_schema', 'ranking_temperature', 'raw_content_limit', 'relevance_weight_core']


----------------------------------------------------------------------
PRIORITY AXES (ranked by importance)
----------------------------------------------------------------------
  1. [HIGH] fuzzy_threshold (pipeline_param) -- step: fuzzy_matching
     Controls candidate inclusion sensitivity
     Values: ['0.6', '0.75', '0.9']
  2. [MEDIUM] fuzzy_scorer (pipeline_param) -- step: fuzzy_matching
     Scorer choice impacts similarity scoring
     Values: ['ratio', 'partial_ratio', 'token_set_ratio']
  3. [HIGH] query_prefix (pipeline_param) -- step: web_search
     Adds context to search queries
     Values: ['"material"', '"LCA"']
  4. [MEDIUM] query_suffix (pipeline_param) -- step: web_search
     Filters results to technical sources
     Values: ['"datasheet"', '"specification"']
  5. [MEDIUM] content_char_limit (pipeline_param) -- step: web_search
     Limits retrieved text for profiling
     Values: ['500', '800', '1200']
  6. [HIGH] profiling_prompt (pipeline_param) -- step: entity

In [12]:
#@title Scan variant config (edit suggested values or add your own)
# Schema axes: mutation tuples ("-", path), ("+", path, type, req, desc),
# ("~", old, new, type, req, desc). Non-schema axes: plain value lists.

scan_sample_size = 10  # queries per scan variant (0 = use all)

scan_variants = {
    'max_token_candidates': [10, 30, 50],
    'query_prefix': ['what material is', 'identify LCA database name for', 'translate trade name'],
    'profiling_schema': [
        [['+', 'geography_scope', 'array', False, 'Relevant ecoinvent/SimaPro geography codes inferred from context, e.g. GLO, RER, CH, RNA'], ['+', 'database_format_hint', 'string', False, "Best-guess ecoinvent-style name fragment for this entity, e.g. 'market for polyethylene, high density'"]],
        # [['-', 'manufacturing_processes'], ['-', 'applications'], ['+', 'lca_synonyms', 'array', False, 'Terms likely to appear verbatim in LCA database entry names for this entity'], ['+', 'no_match_signal', 'string', False, 'Brief reasoning on whether a database match is likely to exist or not']],
        # [['~', 'classification_aliases', 'lca_classification_aliases', 'array', False, 'Expert-level aliases specifically aligned with LCA database naming conventions, including ecoinvent activity names and SimaPro process names'], ['+', 'geography_scope', 'array', False, 'Relevant ecoinvent/SimaPro geography codes inferred from context, e.g. GLO, RER, CH, RNA']],
        [['+', 'lca_database_names', 'array', True, "Likely ecoinvent or GaBi database entry names that would match this entity, using standard LCA database naming conventions like 'market for X | X | cut-off, U'"]], 
        [['-', 'manufacturing_processes'], ['-', 'applications'], ['+', 'lca_database_names', 'array', True, 'Likely ecoinvent or GaBi database entry names for this entity using standard LCA naming conventions']],
        [['~', 'notes', 'material_category', 'string', True, "The broad LCA material category this entity belongs to, e.g. 'polyethylene', 'brass', 'steel'"]]
    ],
    'profiling_temperature': [0.0, 0.3, 0.7],
    # 'profiling_max_tokens': [512, 1024, 2048], # -> Going to cause lots of Errors.
    'raw_content_limit': [1000, 2500, 8000],
}
scan_variants, schema_labels = resolve_scan_variants(scan_variants, svc=svc)

  max_token_candidates: [10, 30, 50]
  query_prefix: ['what material is', 'identify LCA database name for', 'translate trade name']
  profiling_schema: (baseline + 4 mutations)
    [0] (baseline)
    [1] ('+', 'geography_scope', 'array', False, 'Relevant ecoinvent/SimaPro geography codes inferred from context, e.g. GLO, RER, CH, RNA'), ('+', 'database_format_hint', 'string', False, 'Best-guess ecoinvent-style name fragment for this entity, e.g. 'market for polyethylene, high density'')
    [2] ('+', 'lca_database_names', 'array', True, 'Likely ecoinvent or GaBi database entry names that would match this entity, using standard LCA database naming conventions like 'market for X | X | cut-off, U'')
    [3] ('-', 'manufacturing_processes'), ('-', 'applications'), ('+', 'lca_database_names', 'array', True, 'Likely ecoinvent or GaBi database entry names for this entity using standard LCA naming conventions')
    [4] ('~', 'notes', 'material_category', 'string', True, 'The broad LCA material 

In [13]:
#@title Prepare scan baseline
baseline_sp = await prepare_scan_baseline(
    baseline, campaign_config,
    pipeline_params=campaign_config.get("pipeline_params"),
    store=svc["store"], backend_id=svc["backend_id"],
    scan_variants=scan_variants,
)

2026-03-13 15:41:00 INFO     [api.services.search.context] restructure_context_cached: hit (alias group)


  Restructured baseline fields (cached):
    persona: You are a candidate evaluation expert.
    task_intent: Summarize the entity profile, identify its category and key distinguishing featu...
    problem_description: Given an entity_profile_json and a list of candidate matches, produce a concise ...
    instruction: TASK 1: Summarize the profile in 1‑2 sentences, identify entity_category, and li...
    thinking_style: Think step by step.
    answer_format: JSON with keys "reasoning" (string) and "ranked_candidates" (array of objects co...
  Search baseline: a1e9133ac23c (render: 1154 chars)


2026-03-13 15:41:01 INFO     [api.services.search.coverage] build_prompt_result_index: 72 runs -> 15 unique prompts, 3027 total query results



  Historical data: 3027 results across 15 unique prompts
  Baseline alias group: 14889901 ↔ 239cefb8 ↔ 24aeb9e7 ↔ 2c14e76c ↔ 2d9d15f6 ↔ 41f88bae ↔ 44eb12a0 ↔ 4ff72b79 ↔ 61ad2b63 ↔ 82f3e7e2 ↔ 830faccd ↔ 9e4f0633 ↔ adb2589d ↔ aeb18154 ↔ bcdc7b72 ↔ c2a36fe9 ↔ c311136d ↔ cec84ce0 ↔ d018a6dc ↔ e169ed86 (20 prompts linked)
  Matching runs: 71, 3012 cached results

  Scan variant coverage (71 matching runs):
    max_token_candidates     10→4 ✓  30→8 ✓  50→4 ✓  (+7 other)
    query_prefix             what material is→5 ✓  identify LCA database name for→1 ✓  translate trade name→1 ✓  (+4 other)
    profiling_schema         {"properties": {"alternative_names": {"items": {"type": "string"}, "type": "array"}, "applications": {"description": "Direct and derived applications based on product characteristics", "items": {"type": "string"}, "type": "array"}, "classification_aliases": {"description": "Full spectrum of valid ways this entity could be referenced using expert-level terminology, from preci

In [14]:
#@title Sensitivity scan
scan_df, axis_profiles = await sensitivity_scan(
    baseline_sp, scan_variants, eval_data, svc.get("backend_client"),
    sample_size=scan_sample_size,
    store=svc["store"], backend_id=svc["backend_id"],
    pipeline_schema=svc.get("pipeline_schema"),
)

Running sensitivity scan...

  Baseline field values:
    persona: You are a candidate evaluation expert.
    task_intent: Summarize the entity profile, identify its category and key distinguishing featu...
    problem_description: Given an entity_profile_json and a list of candidate matches, produce a concise ...
    instruction: TASK 1: Summarize the profile in 1‑2 sentences, identify entity_category, and li...
    thinking_style: Think step by step.
    answer_format: JSON with keys "reasoning" (string) and "ranked_candidates" (array of objects co...

  Axes: 5, variants: 17, queries/variant: 10, cached results: 11370
  Estimated calls: ~170


2026-03-13 15:41:01 INFO     [httpx] HTTP Request: POST http://127.0.0.1:8000/sessions "HTTP/1.1 200 OK"


  Evaluating baseline...
        MISS 2/20  [token] ⚡  PA66-GF25 ULTRAMID A3UG5 RAL7035 grey          -> Glass fibre reinforced plastic | 95 4.0s
        HIT   [token] ⚡  Stainless steel EN 10270-3/winding             -> Wire drawing, steel {RER}| wire dra 4.1s
        MISS 5/20  [token] ⚡  SJRG0010-ABS/molding                           -> Acrylonitrile-butadiene-styrene cop 4.2s
        MISS --/20  [token] ⚡  Kingfa NPG25                                   -> Polyamide (Nylon) 6.6/EU-27 5.2s
        MISS 13/20  [token] ⚡  PA 66 25% GF V0 RAL 7012/0                     -> Glass fibre reinforced plastic | 95 4.9s
        MISS 6/20  [token] ⚡  PA6/66 Ultramid C3U/molding                    -> Polyamide (Nylon) 6.6/EU-27 4.5s
        MISS 10/20  [token] ⚡  PC  GF10  makrolon material/0                  -> Glass fibre reinforced plastic | 90 3.9s
        MISS 6/20  [token] ⚡  Copper Wire/cold forming                       -> Galvanized Copper | 99.6% Copper 0. 4.0s
        MISS 10/20  [toke

In [15]:
#@title Scan analytics: variant leaderboard
if scan_df is not None and not scan_df.empty:
    show_scan_leaderboard(scan_df, axis_profiles)

VARIANT LEADERBOARD (all scan combos)


,rank,axis,variant,accuracy,delta,hits/total,errors
0,1,max_token_candidates,30,30.0%,+18.0%,3/10,0
1,2,profiling_schema,schema(12 fields),20.0%,+9.2%,2/10,0
2,3,query_prefix,what material is,20.0%,+9.2%,2/10,0
3,4,profiling_temperature,0.3,20.0%,+9.0%,2/10,0
4,5,max_token_candidates,50,20.0%,+9.0%,2/10,0
5,6,raw_content_limit,8000,10.0%,+0.3%,1/10,0
6,7,profiling_schema,schema(11 fields),10.0%,+0.3%,1/10,0
7,8,profiling_schema,schema(11 fields),10.0%,+0.3%,1/10,0
8,9,raw_content_limit,1000,10.0%,+0.3%,1/10,0
9,10,profiling_temperature,0.7,10.0%,+0.3%,1/10,0



PER-AXIS STATISTICS


,axis,type,variants,mean_acc,std_acc,best_acc,worst_acc,sensitivity,budget
0,max_token_candidates,pipeline_param,3,16.7%,15.3%,30.0%,0.0%,0.270,medium
1,query_prefix,pipeline_param,3,6.7%,11.5%,20.0%,0.0%,0.185,medium
2,profiling_schema,pipeline_param,5,10.0%,7.1%,20.0%,0.0%,0.180,medium
3,profiling_temperature,pipeline_param,3,10.0%,10.0%,20.0%,0.0%,0.177,medium
4,raw_content_limit,pipeline_param,3,6.7%,5.8%,10.0%,0.0%,0.090,skip


In [16]:
#@title Scan analytics: query difficulty
if scan_df is not None and not scan_df.empty:
    difficulty_df = show_scan_query_difficulty(
        svc["store"], svc["backend_id"],
    )

QUERY DIFFICULTY (728 queries across 59 scan runs)
  easy: 0 (0%) | discriminating: 5 (1%) | hard: 5 (1%) | error: 718 (99%)



,query,ground_truth,hit_rate,hits/evals,error_rate,classification
0,"BAND EN 10140-1,45x26 GK-DC01+C390-MB","Steel, unalloyed {GLO}| market for steel, unal...",0.000000,0/11,1.000000,error
1,"Kaltband EN 10140-2,5 x ... GK\nStahl EN 10139...","Steel removed by milling, small parts {RER}| s...",0.000000,0/11,1.000000,error
2,Adhesive label 13x5 white\nPolyethylen (PE) we...,"Polyethylene, low density, granulate {GLO}| ma...",0.000000,0/11,1.000000,error
3,"Strip EN13599-CU-PHC-R290-1,8x35-Ag0,3 /stamping","Metal working, average for copper product manu...",0.000000,0/22,1.000000,error
4,PA 66 25% GF V0 RAL 7012/0,Injection moulding {RoW}| injection moulding |...,0.000000,0/70,0.542857,hard
...,...,...,...,...,...,...
723,SJRG0013-PA/molding,Injection moulding {RER}| injection moulding |...,0.025000,1/40,0.550000,discriminating
724,SJRG0010-ABS/molding,Injection moulding {RER}| injection moulding |...,0.050847,3/59,0.457627,discriminating
725,Kingfa NPG25,Glass fibre reinforced plastic | 75% PA66 25% ...,0.220339,13/59,0.457627,discriminating
726,PA66-GF25 ULTRAMID A3UG5 RAL7035 grey,Glass fibre reinforced plastic | 75% PA66 25% ...,0.288136,17/59,0.457627,discriminating


In [17]:
#@title Select scan winner & seed campaign
best_sp = seed_campaign_from_scan(
    scan_df, axis_profiles, baseline_sp, scan_variants,
    campaign_rounds, campaign_config,
)

2026-03-13 15:41:02 INFO     [api.services.search.smart_search] select_scan_winner: 0 prompt changes, 4 param changes from 4 improving axes


Selected best from 4 improving axes:
  max_token_candidates      best_delta=+18.0%  value_idx=1  acc=30.0%
  query_prefix              best_delta=+9.2%  value_idx=0  acc=20.0%
  profiling_schema          best_delta=+9.2%  value_idx=2  acc=20.0%
  profiling_temperature     best_delta=+9.0%  value_idx=1  acc=20.0%
Pipeline params updated: {'steps': ['cache_lookup', 'fuzzy_matching', 'web_search', 'entity_profiling', 'token_matching'], 'max_token_candidates': 30, 'query_prefix': 'what material is', 'profiling_schema': {'type': 'object', 'properties': {'entity_name': {'type': 'string'}, 'core_concept': {'type': 'string', 'description': 'The single word that defines what this expression represents'}, 'distinguishing_features': {'type': 'array', 'items': {'type': 'string'}}, 'key_properties': {'type': 'array', 'items': {'type': 'string'}}, 'technical_specifications': {'type': 'array', 'items': {'type': 'string'}, 'description': 'Explicit technical specs, dimensions, codes, ratings, tolerance

### 3b. Grid Search

<details>
<summary>Skip if you used Smart Search above.</summary>

Systematic sweep of the prompt configuration space. Maps the accuracy landscape before hill-climbing.

</details>

In [18]:
# #@title Grid campaign overview (existing plans)
# merge_plans = False  # Set True to combine results from multiple plans
# grid_overview = show_grid_overview(svc, campaign_config, merge_plans=merge_plans)
# merged_grid_df = grid_overview.get("merged_grid_df")

In [19]:
# #@title Build or resume grid plan
# gs = campaign_config["grid_search"]

# llm_client, llm_model = setup_llm(campaign_config)

# (
#     grid_plan_id, grid_points, grid_state_lookup,
#     grid_axes, layer1_fields, grid_baseline,
# ) = await resume_or_build_grid(
#     campaign_config, baseline, llm_client, llm_model,
#     svc["store"], svc["backend_id"],
#     improvement_areas=campaign_config.get("improvement_areas", ""),
# )

# print(f"Grid points: {len(grid_points)}")
# print(f"Plan ID: {grid_plan_id}")

In [20]:
# #@title Run grid search
# grid_df = await run_grid_search(
#     grid_points, grid_state_lookup, eval_data,
#     campaign_config["eval_llm"],
#     plan_id=grid_plan_id,
#     store=svc["store"], backend_id=svc["backend_id"],
#     backend_client=svc.get("backend_client"),
#     session_terms=svc.get("session_terms"),
#     pipeline_params=campaign_config.get("pipeline_params"),
#     sample_size=gs.get("sample_size", 1),
#     shared_queries=gs.get("shared_queries", False),
#     grid_seed=gs.get("seed", 42),
# )

In [21]:
# #@title Display grid results
# _display_df = merged_grid_df if merged_grid_df is not None else grid_df
# display_grid_results(_display_df, grid_axes, top_k=gs.get("top_k", 5))

In [22]:
# #@title LLM analysis of grid results
# _analysis_df = merged_grid_df if merged_grid_df is not None else grid_df
# llm_client, llm_model = setup_llm(campaign_config)
# grid_analysis = await analyze_grid_results(
#     _analysis_df, grid_axes, llm_client, model=llm_model,
# )

In [23]:
# #@title Select grid winner and seed campaign
# grid_winner = select_and_seed_grid_winner(
#     grid_df, merged_grid_df, grid_state_lookup,
#     grid_overview.get("plan_dfs", {}), svc, campaign_rounds,
# )

## 4. Optimize

Two modes: **Semi-automatic** (feedback cycle with patience-based auto-stop) or **Manual** (one round at a time).

In [24]:
#@title Feedback cycle preflight
scan_context = show_feedback_preflight(
    campaign_rounds, eval_data, campaign_config,
    pipeline_params=campaign_config.get("pipeline_params"),
    scan_df=scan_df if "scan_df" in dir() else None,
    axis_profiles=axis_profiles if "axis_profiles" in dir() else None,
    scan_variants=scan_variants if "scan_variants" in dir() else None,
    difficulty_df=difficulty_df if "difficulty_df" in dir() else None,
)


  FEEDBACK CYCLE PRE-FLIGHT
  Baseline accuracy      : 10.0%
  Baseline prompt        : TASK 1: Summarize the profile in 1‑2 sentences, identify entity_category, and li...
  ------------------------------------------------------------------
  Max rounds             : 3
  Candidates per round   : 5
  Queries per eval       : 15 of 984
  Improvement threshold  : 1.0%
  Patience (L1)          : 2 rounds
  L2 (refine context)    : disabled
  L3 (modify plan)       : disabled
  ------------------------------------------------------------------
  Candidate model        : openai/gpt-oss-120b
  Creativity             : 0.7
  Pipeline               : 5 of 6 steps
    Steps                : cache_lookup, fuzzy_matching, web_search, entity_profiling, token_matching
    Excluded             : llm_ranking
  Strategy               : SCAN-AWARE

  ROUND PIPELINE (what happens each round)
  ------------------------------------------------------------------
  1. BASELINE INPUT
     Prompt: TASK 1: Sum

In [26]:
#@title Run optimization (feedback cycle)
campaign_rounds = await run_feedback_cycle_notebook(
    campaign_rounds, eval_data, campaign_config,
    store=svc["store"], backend_id=svc["backend_id"],
    backend_url=svc["backend_client"].base_url,
    pipeline_params=campaign_config.get("pipeline_params"),
    session_terms=svc.get("session_terms"),
    scan_context=scan_context if "scan_context" in dir() else None,
)

2026-03-13 15:48:54 INFO     [api.services.campaign.feedback_cycle] Using provided baseline (acc=0.200)
2026-03-13 15:48:54 INFO     [api.services.campaign.feedback_cycle] Cycle identity: cycle_68e2c53845c3



╔════════════════════════════════════════════════════════════════════╗
║  FEEDBACK CYCLE STARTING                                           ║
╠════════════════════════════════════════════════════════════════════╣
║  Baseline       20.0%                                              ║
║  Max rounds     3              Patience    2                       ║
║  Candidates     5                                                  ║
║  Sample size    15 of 984                                          ║
║  Min detectable ±36.2% (α=0.05, 80% power)                         ║
║  Model          openai/gpt-oss-120b                                ║
║  L2 (refine)    disabled           L3 (plan)   disabled            ║
║  Scan context   YES                                                ║
╚════════════════════════════════════════════════════════════════════╝


2026-03-13 15:48:54 INFO     [api.services.campaign.feedback_cycle] Resuming cycle cycle_68e2c53845c3 from round 1 (best_acc=0.200)
2026-03-13 15:48:55 INFO     [api.services.obs.observability_logger] Dataset 'termnorm_ground_truth': 728 items registered, 256 duplicates/empty skipped (from 984 input)
2026-03-13 15:48:55 WARNING  [api.services.obs.observability_logger] Skipping Langfuse cloud dataset registration for 984 items (rate-limit risk). Use the dedicated Langfuse sync cell instead.
2026-03-13 15:48:55 INFO     [api.services.campaign.feedback_cycle] Registered 728 dataset items for 'termnorm_ground_truth'
2026-03-13 15:48:55 INFO     [api.services.campaign.feedback_cycle] Feedback cycle round 0 (acc=0.200, stall=0/2)
2026-03-13 15:48:55 INFO     [api.services.campaign.feedback_cycle] Loaded 5 persisted candidates for round 0


  ✓ Initialized  cycle=cycle_68e2c5  samples=15  obs=ON
    Resumed from round 1 (1 rounds cached)

┌─ Round 1/3 ───────────────────────────────────────── patience 0/2 ─┐
│  GENERATING CANDIDATES                                             │
│  Current best    20.0%                                             │
│  Prompt          You are a candidate evaluation expert.  Summari...│
│  Candidates      5   Creativity: 0.7   Scan: YES                   │
│  Model           openai/gpt-oss-120b                               │
└────────────────────────────────────────────────────────────────────┘
  Scan focus: 4 improving axes [max_token_candidates, query_prefix, profiling_schema, profiling_temperature]
  Scan baseline: 10.0%
  ✓ 5 candidates generated (loaded from disk)
    C1: Increase max_token_candidates to 40 and set pro... pp=[max_token_candidates, profiling_temperature, query_prefix, +1]
    C2: Swap query_prefix to a semantic variant and exp... pp=[query_prefix, profiling_schema, max_

2026-03-13 15:48:55 INFO     [api.services.campaign.feedback_cycle] Feedback cycle round 1 (acc=0.200, stall=1/2)
2026-03-13 15:48:55 INFO     [api.services.campaign.feedback_cycle] Loaded 5 persisted candidates for round 1



  ┌─ C5/5 ───────────────────────────────────── 6.7% [1.2%-29.8%] ─┐
  │  Explore a larger token budget (70) and hig...  pp=[max_token_candidates, profiling_temperature, +3]│
  │  1/15 hits  composite=0.0600  vs baseline: -13.3%              │
  │  best so far: C4 20.0%                                         │
  └────────────────────────────────────────────────────────────────┘
  ┌─ SCOREBOARD ───────────────────────────────────────────────────────────────┐
  │  #   Label    Accuracy            95% CI  Composite    Delta               │
  │  1   C4         20.0%       [7.0%-45.2%]     0.1800       ---  *           │
  │  2   C3         13.3%       [3.7%-37.9%]     0.1200     -6.7%              │
  │  3   C1          6.7%       [1.2%-29.8%]     0.0600    -13.3%              │
  │  4   C2          6.7%       [1.2%-29.8%]     0.0600    -13.3%              │
  │  5   C5          6.7%       [1.2%-29.8%]     0.0600    -13.3%              │
  └───────────────────────────────────────────────

2026-03-13 15:48:55 INFO     [api.services.prompt_eval] Resuming candidate_4_b9f19189: 1 cached results, 14 remaining



  ┌─ C4/5 ───────────────────────────────────── 0.0% [0.0%-20.4%] ─┐
  │  Lower profiling_temperature slightly  pp=[profiling_temperature]│
  │  0/15 hits  ⚠ 6/15 degraded  vs baseline: -20.0%               │
  │  best so far: C2 13.3%                                         │
  └────────────────────────────────────────────────────────────────┘
  [ 61] HIT   [token] ⚡  PA66-GF25 ULTRAMID A3UG5 RAL7035 grey          -> Glass fibre reinforced plastic | 75 7.2s
  [ 62] HIT   [token]  Stainless steel EN 10270-3/winding             -> Wire drawing, steel {RER}| wire dra 5.9s
  [ 63] MISS --/20  [token]  SJRG0010-ABS/molding                           -> Acrylonitrile-butadiene-styrene cop 8.3s
                  ⚠ web_search: 2 of 14 fetched URLs returned content
  [ 64] MISS 4/20  [token]  Kingfa NPG25                                   -> Glass fibre reinforced plastic | 50 21.1s
                  ⚠ web_search: 3 of 14 fetched URLs returned content
  [ 65] MISS 11/20  [token]  PA 66 25% GF 

2026-03-13 15:51:11 INFO     [api.services.campaign.feedback_cycle] Patience exhausted after 2 stalls at round 1



  ┌─ C5/5 ──────────────────────────────────── 13.3% [3.7%-37.9%] ─┐
  │  Increase raw_content_limit to allow more c...  pp=[raw_content_limit]│
  │  2/15 hits  composite=0.1200  ⚠ 9/15 degraded  vs baseline: -6.7%│
  │  best so far: C2 13.3%                                         │
  └────────────────────────────────────────────────────────────────┘
  ┌─ SCOREBOARD ───────────────────────────────────────────────────────────────┐
  │  #   Label    Accuracy            95% CI  Composite    Delta               │
  │  1   C2         13.3%       [3.7%-37.9%]     0.1200     -6.7%  *           │
  │  2   C5         13.3%       [3.7%-37.9%]     0.1200     -6.7%              │
  │  3   C3          6.7%       [1.2%-29.8%]     0.0600    -13.3%              │
  │  4   C1          0.0%       [0.0%-20.4%]     0.0000    -20.0%              │
  │  5   C4          0.0%       [0.0%-20.4%]     0.0000    -20.0%              │
  └───────────────────────────────────────────────────────────────────────────

In [ ]:
#@title Run optimization round (manual)
round_entry = await run_manual_round(
    campaign_rounds, eval_data, campaign_config, svc,
)

## 5. Results

In [ ]:
#@title Campaign comparison table
show_campaign_summary(campaign_rounds)

In [ ]:
#@title Per-query flip tracking (baseline vs final)
show_flip_tracking(campaign_rounds)

In [ ]:
#@title PromptState lineage chain
show_lineage_chain(campaign_rounds)

In [ ]:
#@title Save winner
save_campaign_winner(campaign_rounds, campaign_config, svc["store"], svc["backend_id"])

In [ ]:
#@title Generate LLM suggestions for next round
llm_client, llm_model = setup_llm(campaign_config)
suggestions = await generate_suggestions(
    campaign_rounds, eval_data, campaign_config,
    llm_client, model=llm_model,
)
display_suggestions(suggestions, len(campaign_rounds))
print("--- SUGGESTED CONFIG (copy to Setup) ---")
print(json.dumps(suggestions.get("suggested_config", campaign_config), indent=2))

In [ ]:
#@title Sync evaluation history to Langfuse
# Safe to re-run — already-pushed runs are skipped automatically.
stats = sync_langfuse(
    svc["store"], svc["backend_id"],
    dataset_name="termnorm_ground_truth",
)